In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [ ]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

train_data.head()

In [ ]:
train_data.shape

In [ ]:
val_data.shape

In [ ]:
# have to preprocess the data, like remove the unwanted elements like HTML tags, new line elements.

## Random Sampling

In [ ]:
train_data = train_data.sample(n = 4000, random_state = 42).reset_index(drop = True)
val_data = val_data.sample(n = 500, random_state = 42).reset_index(drop = True)

In [ ]:
train_data.shape

# Data Preprocessing

In [ ]:
import re

def clean_data(text):
    text = re.sub(r"\r\n"," ", text)  # line replacement
    text = re.sub(r"\s+"," ", text)  # empty space replacement
    text = re.sub(r"<.*?>"," ", text)  # HTML tag replacement
    text = text.strip().lower()

    return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)


val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [ ]:
train_data["dialogue"][0]

# Tokenization

In [ ]:
# tokenization means converting our text into numbers
# like for "I ate an apple" --> 32,405,111,727

In [ ]:
# We will be using T5Tokenizer for T5 model and it has ~32k tokens in its vocabulary.

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:
# function for raw data -> tokens for fine-tuning of our pre-trained model

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding = "max_length", max_length = 512, truncation = True)
    targets = tokenizer(data["summary"], padding = "max_length", max_length = 150, truncation = True)

    inputs["labels"] = targets["input_ids"] # token id => add to input as labels
    return inputs

In [ ]:
train_df = train_data.apply(tokenize, axis = 1).tolist()
val_df = val_data.apply(tokenize, axis = 1).tolist()

In [ ]:
train_df[0]

In [ ]:
## input_ids = dialogue -> token_id


# 1 -> EOS  # 0 -> padding


# attention_mask -> it tells us the valid values in the input_ids; 1 -> valid

# labels --> targets(summary)

print(len(train_df[0]["input_ids"]))
print(len(train_df[0]["labels"]))
print(type(train_df))
print(type(val_df))

# Fine Tuning our Pre-trained Model

In [ ]:
# here we are summarizing our text. So, it's a NLP/generation task.
# now this generation is dependent on the input. So, it is conditional.

model = T5ForConditionalGeneration.from_pretrained("t5-small")

In [ ]:
# Fine-Tune

import torch

if torch.backends.mps.is_available():
    device = torch.device("nps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device is {device}")
model.to(device)

In [ ]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",     # want to validate our model after every epoch
    save_strategy = "epoch",     # want to save our model after every epoch

    warmup_steps = 500
    # in this our learning rate increase from 0 to its default value in 500 steps.
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_df,
    eval_dataset = val_df
)

In [ ]:
# Train the model

trainer.train()

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

# Testing the core logic for Summarization

In [ ]:
def summarize_dialogue(dialogue):

  # Data Preprocessing

  dialogue = clean_data(dialogue)

  # generate tokens of the dialogue
  inputs = tokenizer(
      dialogue,
      padding = "max_length",
      max_length = 512,
      truncation = True,
      return_tensors = "pt"
  ).to(device)

  # generate summary -> which will be in tokens
  model.to(device)
  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length = 150,
      num_beams = 4,   # transformer will generate 4 diff. o/p and give us the best one among those 4 o/p.
      early_stopping = True
  )



  # tokens id convertion into text => decoding

  summary = tokenizer.decode(targets[0], skip_special_tokens = True)

  return summary


In [ ]:
test_dialogue = """Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""


summary = summarize_dialogue(test_dialogue)

print(f"Summary is :\n {summary}")